# Qdrant Migration

**Use case:** Same migration story as ChromaDB, but on Qdrant. Demonstrates that isotrieve's adapter pattern works uniformly across vector store backends.

**When you'd reach for this:** You use Qdrant and want to migrate to a new embedding model without re-embedding.

**What you need installed:** `isotrieve`, `qdrant-client`, `numpy`.

**Estimated runtime:** ~2 minutes.

**This notebook focuses on:** What changes in the adapter code vs. Chroma — the migration concept is the same, the API surface differs.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/AECP/blob/main/isotrieve-python/notebooks/03_qdrant_migration.ipynb)

In [ ]:
!pip install -q isotrieve qdrant-client numpy scikit-learn

In [ ]:
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from isotrieve import RidgeMapping
from isotrieve.adapters.qdrant import QdrantAdapter

print("imports OK")

## 1. Set up an in-memory Qdrant instance

We use `QdrantClient(":memory:")` so no server is needed. In production, you'd point to your Qdrant instance.

In [ ]:
rng = np.random.default_rng(42)

N_DOCS = 300
D_OLD = 384
D_NEW = 768
LATENT = 64

latent = rng.normal(size=(N_DOCS, LATENT))
W_old = rng.normal(size=(LATENT, D_OLD)) / np.sqrt(LATENT)
W_new = rng.normal(size=(LATENT, D_NEW)) / np.sqrt(LATENT)

doc_vectors_old = latent @ W_old
doc_vectors_old = doc_vectors_old / np.linalg.norm(doc_vectors_old, axis=1, keepdims=True)

doc_vectors_new = latent @ W_new
doc_vectors_new = doc_vectors_new / np.linalg.norm(doc_vectors_new, axis=1, keepdims=True)

# Create Qdrant collection with old-model vectors
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=D_OLD, distance=Distance.COSINE),
)

points = [
    PointStruct(id=i, vector=doc_vectors_old[i].tolist(), payload={"text": f"Doc {i}"})
    for i in range(N_DOCS)
]
client.upsert(collection_name="docs", points=points)

info = client.get_collection("docs")
print(f"Qdrant collection 'docs': {info.points_count} vectors, dim={D_OLD}")

## 2. Fit the mapping

In [ ]:
N_CAL = 1500
latent_cal = rng.normal(size=(N_CAL, LATENT))
X_cal = (latent_cal @ W_old)
X_cal = X_cal / np.linalg.norm(X_cal, axis=1, keepdims=True)
Y_cal = (latent_cal @ W_new)
Y_cal = Y_cal / np.linalg.norm(Y_cal, axis=1, keepdims=True)

mapping = RidgeMapping(alpha="auto", seed=0)
mapping.fit(X_cal, Y_cal)
print(f"Mapping: {mapping.d_src} -> {mapping.d_tgt}, alpha={mapping.validation_report().alpha:.2f}")

## 3. Create the adapter and migrate

The `QdrantAdapter` wraps the mapping + Qdrant client. The `migrate()` method scrolls through the source collection, transforms each vector, and writes to a new collection.

In [ ]:
from unittest.mock import patch
from isotrieve.adapters.qdrant import QdrantAdapter

# The QdrantAdapter constructor tries QdrantClient(url=...), which fails with ":memory:"
# We mock _require_qdrant to return our existing in-memory client
fake_factory = lambda url, api_key=None: client
with patch("isotrieve.adapters.qdrant._require_qdrant", return_value=(fake_factory, PointStruct)):
    adapter = QdrantAdapter(mapping, url=":memory:", collection="docs")

report = adapter.migrate(new_collection="docs_new")

print(f"Migrated: {report.rows_processed} vectors")
print(f"Source: {report.source_collection} -> Target: {report.target_collection}")

new_info = client.get_collection("docs_new")
print(f"Target collection: {new_info.points_count} vectors, dim={D_NEW}")

## 4. Serve-mode queries

In serve mode, queries come from the **new model** (d_tgt) and are inverse-mapped to the old model's space to search the source collection. No corpus modification needed.

In [ ]:
# Queries from the new model
N_QUERIES = 5
latent_q = rng.normal(size=(N_QUERIES, LATENT))
Q_new = (latent_q @ W_new)
Q_new = Q_new / np.linalg.norm(Q_new, axis=1, keepdims=True)

# Search via serve mode (adapter inverse-maps queries automatically)
results = adapter.query(Q_new, k=5)

for i, hits in enumerate(results):
    top_ids = [h["id"] for h in hits]
    top_scores = [f"{h['score']:.3f}" for h in hits]
    print(f"Query {i}: top-5 ids={top_ids[:3]}... scores={top_scores[:3]}...")

## 5. Compare ChromaDB vs Qdrant adapter patterns

The migration concept is identical; only the backend-specific API changes:

In [ ]:
comparison = """
┌─────────────────────┬──────────────────────────┬──────────────────────────┐
│                     │ ChromaDB                 │ Qdrant                   │
├─────────────────────┼──────────────────────────┼──────────────────────────┤
│ Client setup        │ chromadb.Client()        │ QdrantClient(url)        │
│ Collection create   │ client.create_collection │ client.create_collection │
│                     │   (name, metadata)       │   (name, vectors_config) │
│ Add vectors         │ collection.add(          │ client.upsert(           │
│                     │   ids, embeddings)       │   collection, points)    │
│ Migration           │ migrate_collection(      │ adapter.migrate(         │
│                     │   client, name, mapping) │   new_collection=name)   │
│ Query-time shim     │ IsotrieveChromaFunction │ adapter.query(           │
│                     │   (mapping, embedder)    │   query_vectors, k)     │
│ Mapping object      │ Same RidgeMapping        │ Same RidgeMapping        │
└─────────────────────┴──────────────────────────┴──────────────────────────┘

Key point: the mapping object is identical in both cases.
isotrieve is backend-agnostic middleware.
"""
print(comparison)

## Try it yourself

Try creating an adapter for the migrated collection and querying it directly with new-model vectors:

```python
with patch("isotrieve.adapters.qdrant._require_qdrant", return_value=(fake_factory, PointStruct)):
    adapter_migrated = QdrantAdapter(mapping, url=":memory:", collection="docs_new", mode="migrated")
results = adapter_migrated.query(Q_new, k=5)
```